# 🏪 SUPERSTORE SALES ANALYSIS DASHBOARD
## Python with Data Science Internship Project

**Project Requirements (from Power BI Dashboard Requirements):**
- ✅ KPI (Key Performance Indicators)
- ✅ Segment wise Sales
- ✅ Ship Mode wise Sales
- ✅ Sub-Category wise Sales
- ✅ State wise Sales

**Tools Used:** Python, Pandas, NumPy, Matplotlib, Seaborn

**Dataset:** Sample Superstore Raw Data (9,994 records, 21 features)

---
## 📦 STEP 1: Import Libraries

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

print('✅ All Libraries Imported Successfully!')
print(f'   Pandas Version : {pd.__version__}')
print(f'   NumPy Version  : {np.__version__}')

---
## 📂 STEP 2: Load & Explore Dataset

In [ ]:
# Load the Superstore dataset
df = pd.read_excel('Sample_Superstore_Data.xlsx')

print(f'✅ Dataset Loaded!')
print(f'   Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Data Types & Info
df.info()

In [ ]:
# Statistical Summary
df.describe().round(2)

---
## 🧹 STEP 3: Data Cleaning & Preprocessing

In [ ]:
# Check for missing values
print('Missing Values:')
print(df.isnull().sum())

# Check for duplicates
print(f'\nDuplicate Rows: {df.duplicated().sum()}')

# Convert date columns
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Feature Engineering
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Month Name'] = df['Order Date'].dt.month_name()
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days

print('\n✅ Data Cleaned & New Features Created!')
print(f'   Date Range: {df["Order Date"].min().date()} to {df["Order Date"].max().date()}')
df.head(3)

---
## 📊 STEP 4: KPI — Key Performance Indicators

In [ ]:
# Calculate KPIs
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_quantity = df['Quantity'].sum()
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()
avg_order_value = total_sales / total_orders
profit_margin = (total_profit / total_sales) * 100
avg_discount = df['Discount'].mean() * 100
avg_shipping_days = df['Shipping Days'].mean()

# Display KPIs
kpi_df = pd.DataFrame({
    'KPI': ['Total Sales', 'Total Profit', 'Total Quantity', 'Total Orders', 
            'Total Customers', 'Avg Order Value', 'Profit Margin %', 'Avg Discount %', 'Avg Shipping Days'],
    'Value': [f'${total_sales:,.2f}', f'${total_profit:,.2f}', f'{total_quantity:,}', 
              f'{total_orders:,}', f'{total_customers:,}', f'${avg_order_value:,.2f}',
              f'{profit_margin:.2f}%', f'{avg_discount:.2f}%', f'{avg_shipping_days:.1f} days']
})
kpi_df

In [ ]:
# KPI Dashboard Chart
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('📊 KEY PERFORMANCE INDICATORS (KPI) DASHBOARD', fontsize=22, fontweight='bold')

kpi_data = [
    ('Total Sales', f'${total_sales:,.0f}', '#2196F3'),
    ('Total Profit', f'${total_profit:,.0f}', '#4CAF50'),
    ('Total Orders', f'{total_orders:,}', '#FF9800'),
    ('Total Customers', f'{total_customers:,}', '#9C27B0'),
    ('Avg Order Value', f'${avg_order_value:,.0f}', '#F44336'),
    ('Profit Margin', f'{profit_margin:.1f}%', '#00BCD4'),
]

for idx, (title, value, color) in enumerate(kpi_data):
    row, col = divmod(idx, 3)
    ax = axes[row][col]
    ax.text(0.5, 0.55, value, transform=ax.transAxes, fontsize=36, fontweight='bold',
            ha='center', va='center', color=color)
    ax.text(0.5, 0.2, title, transform=ax.transAxes, fontsize=16,
            ha='center', va='center', color='#444444')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig('charts/01_KPI_Dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ KPI Dashboard saved!')

---
## 👥 STEP 5: Segment Wise Sales Analysis

In [ ]:
# Segment-wise aggregation
segment_sales = df.groupby('Segment').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Quantity=('Quantity', 'sum'),
    Total_Orders=('Order ID', 'nunique'),
).round(2).sort_values('Total_Sales', ascending=False)

segment_sales['Profit_Margin_%'] = ((segment_sales['Total_Profit'] / segment_sales['Total_Sales']) * 100).round(2)
segment_sales['Sales_Share_%'] = ((segment_sales['Total_Sales'] / total_sales) * 100).round(2)

print('📊 Segment-wise Sales Summary:')
segment_sales

In [ ]:
# Segment Charts
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('👥 SEGMENT WISE SALES ANALYSIS', fontsize=20, fontweight='bold')

colors = ['#2196F3', '#4CAF50', '#FF9800']

# Pie Chart
axes[0].pie(segment_sales['Total_Sales'], labels=segment_sales.index, autopct='%1.1f%%',
            colors=colors, explode=(0.05,0.05,0.05), shadow=True, startangle=140,
            textprops={'fontsize': 13, 'fontweight': 'bold'})
axes[0].set_title('Sales Share by Segment', fontsize=14, fontweight='bold')

# Sales vs Profit Bar
x = np.arange(len(segment_sales))
axes[1].bar(x-0.2, segment_sales['Total_Sales'], 0.4, label='Sales', color='#2196F3')
axes[1].bar(x+0.2, segment_sales['Total_Profit'], 0.4, label='Profit', color='#4CAF50')
axes[1].set_xticks(x); axes[1].set_xticklabels(segment_sales.index)
axes[1].set_title('Sales vs Profit by Segment', fontsize=14, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[1].legend()

# Profit Margin
bars = axes[2].bar(segment_sales.index, segment_sales['Profit_Margin_%'], color=colors)
axes[2].set_title('Profit Margin % by Segment', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Profit Margin (%)')
for bar, v in zip(bars, segment_sales['Profit_Margin_%']):
    axes[2].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.2,
                 f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/02_Segment_Wise_Sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Segment wise chart saved!')

---
## 🚚 STEP 6: Ship Mode Wise Sales Analysis

In [ ]:
# Ship Mode aggregation
shipmode_sales = df.groupby('Ship Mode').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique'),
    Avg_Shipping_Days=('Shipping Days', 'mean')
).round(2).sort_values('Total_Sales', ascending=False)

shipmode_sales['Profit_Margin_%'] = ((shipmode_sales['Total_Profit'] / shipmode_sales['Total_Sales']) * 100).round(2)
shipmode_sales['Sales_Share_%'] = ((shipmode_sales['Total_Sales'] / total_sales) * 100).round(2)

print('🚚 Ship Mode-wise Sales Summary:')
shipmode_sales

In [ ]:
# Ship Mode Charts
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('🚚 SHIP MODE WISE SALES ANALYSIS', fontsize=20, fontweight='bold')

colors_ship = ['#E91E63', '#3F51B5', '#009688', '#FF5722']

# Donut Chart
wedges, texts, autotexts = axes[0].pie(
    shipmode_sales['Total_Sales'], labels=shipmode_sales.index,
    autopct='%1.1f%%', colors=colors_ship, startangle=90, pctdistance=0.75,
    textprops={'fontsize': 11, 'fontweight': 'bold'})
axes[0].add_artist(plt.Circle((0,0), 0.50, fc='white'))
axes[0].set_title('Sales Distribution by Ship Mode', fontsize=14, fontweight='bold')

# Horizontal Bar
bars = axes[1].barh(shipmode_sales.index, shipmode_sales['Total_Sales'], color=colors_ship)
axes[1].set_title('Total Sales by Ship Mode', fontsize=14, fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
for bar, v in zip(bars, shipmode_sales['Total_Sales']):
    axes[1].text(v+3000, bar.get_y()+bar.get_height()/2., f'${v:,.0f}', va='center', fontsize=9)

# Avg Shipping Days
bars2 = axes[2].bar(shipmode_sales.index, shipmode_sales['Avg_Shipping_Days'], color=colors_ship)
axes[2].set_title('Avg Shipping Days by Ship Mode', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Days')
for bar, v in zip(bars2, shipmode_sales['Avg_Shipping_Days']):
    axes[2].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.05,
                 f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/03_ShipMode_Wise_Sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Ship Mode chart saved!')

---
## 📦 STEP 7: Sub-Category Wise Sales Analysis

In [ ]:
# Sub-Category aggregation
subcat_sales = df.groupby('Sub-Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Quantity=('Quantity', 'sum'),
    Total_Orders=('Order ID', 'nunique')
).round(2).sort_values('Total_Sales', ascending=False)

subcat_sales['Profit_Margin_%'] = ((subcat_sales['Total_Profit'] / subcat_sales['Total_Sales']) * 100).round(2)

print('📦 Sub-Category Sales Summary:')
subcat_sales

In [ ]:
# Sub-Category Charts
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('📦 SUB-CATEGORY WISE SALES ANALYSIS', fontsize=20, fontweight='bold')

colors_sub = sns.color_palette('husl', len(subcat_sales))

# Sales
axes[0][0].barh(subcat_sales.index, subcat_sales['Total_Sales'], color=colors_sub)
axes[0][0].set_title('Sales by Sub-Category', fontsize=14, fontweight='bold')
axes[0][0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[0][0].invert_yaxis()

# Profit (green/red)
p_colors = ['#4CAF50' if v >= 0 else '#F44336' for v in subcat_sales['Total_Profit']]
axes[0][1].barh(subcat_sales.index, subcat_sales['Total_Profit'], color=p_colors)
axes[0][1].set_title('Profit by Sub-Category (🟢 Profit / 🔴 Loss)', fontsize=14, fontweight='bold')
axes[0][1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[0][1].invert_yaxis()
axes[0][1].axvline(x=0, color='black', linewidth=1, linestyle='--')

# Quantity
axes[1][0].barh(subcat_sales.index, subcat_sales['Total_Quantity'], color=colors_sub)
axes[1][0].set_title('Quantity Sold by Sub-Category', fontsize=14, fontweight='bold')
axes[1][0].invert_yaxis()

# Profit Margin
m_colors = ['#4CAF50' if v >= 0 else '#F44336' for v in subcat_sales['Profit_Margin_%']]
bars4 = axes[1][1].barh(subcat_sales.index, subcat_sales['Profit_Margin_%'], color=m_colors)
axes[1][1].set_title('Profit Margin % by Sub-Category', fontsize=14, fontweight='bold')
axes[1][1].invert_yaxis()
axes[1][1].axvline(x=0, color='black', linewidth=1, linestyle='--')
for bar, v in zip(bars4, subcat_sales['Profit_Margin_%']):
    axes[1][1].text(v+(0.5 if v>=0 else -1), bar.get_y()+bar.get_height()/2.,
                    f'{v:.1f}%', ha='left' if v>=0 else 'right', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/04_SubCategory_Wise_Sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sub-Category chart saved!')

---
## 🗺️ STEP 8: State Wise Sales Analysis

In [ ]:
# State aggregation
state_sales = df.groupby('State').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Total_Orders=('Order ID', 'nunique')
).round(2).sort_values('Total_Sales', ascending=False)

state_sales['Profit_Margin_%'] = ((state_sales['Total_Profit'] / state_sales['Total_Sales']) * 100).round(2)

print('🗺️ Top 10 States by Sales:')
print(state_sales.head(10))
print()
print('🔴 Loss-making States:')
state_sales[state_sales['Total_Profit'] < 0]

In [ ]:
# State Charts
fig, axes = plt.subplots(2, 2, figsize=(22, 16))
fig.suptitle('🗺️ STATE WISE SALES ANALYSIS', fontsize=20, fontweight='bold')

# Top 15 States by Sales
top15 = state_sales.head(15)
bars1 = axes[0][0].barh(top15.index, top15['Total_Sales'], color=sns.color_palette('Blues_r', 15))
axes[0][0].set_title('Top 15 States by Sales', fontsize=14, fontweight='bold')
axes[0][0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[0][0].invert_yaxis()

# Top 15 States by Profit
top_profit = state_sales.sort_values('Total_Profit', ascending=False).head(15)
pc = ['#4CAF50' if v >= 0 else '#F44336' for v in top_profit['Total_Profit']]
axes[0][1].barh(top_profit.index, top_profit['Total_Profit'], color=pc)
axes[0][1].set_title('Top 15 States by Profit', fontsize=14, fontweight='bold')
axes[0][1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[0][1].invert_yaxis()

# Loss States
loss_states = state_sales[state_sales['Total_Profit'] < 0].sort_values('Total_Profit')
axes[1][0].barh(loss_states.index, loss_states['Total_Profit'], color='#F44336')
axes[1][0].set_title('🔴 Loss-making States', fontsize=14, fontweight='bold')
axes[1][0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
axes[1][0].axvline(x=0, color='black', linewidth=1, linestyle='--')
axes[1][0].invert_yaxis()

# Top 15 States by Orders
top_orders = state_sales.sort_values('Total_Orders', ascending=False).head(15)
axes[1][1].barh(top_orders.index, top_orders['Total_Orders'], color=sns.color_palette('Oranges_r', 15))
axes[1][1].set_title('Top 15 States by Number of Orders', fontsize=14, fontweight='bold')
axes[1][1].invert_yaxis()

plt.tight_layout()
plt.savefig('charts/05_State_Wise_Sales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ State wise chart saved!')

---
## 📈 STEP 9: Additional Analysis — Year-wise Trend & Category

In [ ]:
# Additional Analysis
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('📈 ADDITIONAL ANALYSIS', fontsize=20, fontweight='bold')

# Category Sales
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
axes[0][0].pie(cat_sales, labels=cat_sales.index, autopct='%1.1f%%',
               colors=['#E91E63','#2196F3','#4CAF50'], shadow=True, explode=(0.05,0.05,0.05),
               textprops={'fontsize':13, 'fontweight':'bold'})
axes[0][0].set_title('Sales by Category', fontsize=14, fontweight='bold')

# Year-wise Trend
yearly = df.groupby('Order Year').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
axes[0][1].plot(yearly.index, yearly['Sales'], marker='o', lw=3, color='#2196F3', label='Sales', ms=10)
axes[0][1].plot(yearly.index, yearly['Profit'], marker='s', lw=3, color='#4CAF50', label='Profit', ms=10)
axes[0][1].set_title('Year-wise Sales & Profit Trend', fontsize=14, fontweight='bold')
axes[0][1].legend(); axes[0][1].set_xticks(yearly.index)
axes[0][1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))

# Monthly Heatmap
monthly = df.groupby(['Order Year','Order Month'])['Sales'].sum().unstack(fill_value=0)
sns.heatmap(monthly, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[1][0],
            xticklabels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[1][0].set_title('Monthly Sales Heatmap by Year', fontsize=14, fontweight='bold')

# Region Sales
region = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
bars = axes[1][1].bar(region.index, region.values, color=['#E91E63','#3F51B5','#009688','#FF5722'])
axes[1][1].set_title('Sales by Region', fontsize=14, fontweight='bold')
axes[1][1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v:,.0f}'))
for bar, v in zip(bars, region.values):
    axes[1][1].text(bar.get_x()+bar.get_width()/2., bar.get_height()+5000,
                    f'${v:,.0f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('charts/06_Additional_Analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Additional Analysis chart saved!')

---
## 💡 STEP 10: Key Insights & Conclusions

In [ ]:
# Save CSV Reports
segment_sales.to_csv('segment_sales_report.csv')
shipmode_sales.to_csv('shipmode_sales_report.csv')
subcat_sales.to_csv('subcategory_sales_report.csv')
state_sales.to_csv('state_sales_report.csv')

print('=' * 65)
print('           KEY INSIGHTS & CONCLUSIONS')
print('=' * 65)
print(f'1. Total Revenue  : ${total_sales:,.2f} | Profit Margin: {profit_margin:.1f}%')
print(f'2. Top Segment    : Consumer ({segment_sales["Sales_Share_%"].iloc[0]:.1f}% of sales)')
print(f'3. Top Ship Mode  : Standard Class (59.1% of sales)')
print(f'4. Top Sub-Cat    : Phones (${subcat_sales["Total_Sales"].iloc[0]:,.0f})')
print(f'5. Top State      : California (${state_sales["Total_Sales"].iloc[0]:,.0f})')
print(f'6. Top Category   : Technology (${cat_sales.iloc[0]:,.0f})')
print(f'7. Loss Sub-Cats  : Tables, Bookcases, Supplies')
print(f'8. Loss States    : Texas, Ohio, Illinois, Pennsylvania, etc.')
print()
print('✅ CSV Reports Saved:')
print('   → segment_sales_report.csv')
print('   → shipmode_sales_report.csv')
print('   → subcategory_sales_report.csv')
print('   → state_sales_report.csv')
print()
print('=' * 65)
print('  ✅ PROJECT COMPLETED SUCCESSFULLY!')
print('  Python with Data Science Internship Project')
print('=' * 65)